# Aprendizado de Máquina — Lista prática 07

## Classificação e Classificadores Gaussianos

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Nesta lista a população é **inteiramente conhecida**: duas gaussianas em $\mathbb{R}^2$
com médias e covariâncias que nós escolhemos. Isso permite algo que num banco real
nunca é possível — calcular o **erro de Bayes**, o piso que nenhum classificador
consegue furar, e medir a que distância dele cada método chega.

> **o QDA é o modelo *correto* para esta população. A lista mede o que ele ganha
> por isso, e quanto de dado ele precisa para cobrar o prêmio.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots
from scipy.stats import multivariate_normal

import sklearn.model_selection as skm
from sklearn.datasets import load_breast_cancer
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — a população e o erro de Bayes

Duas classes gaussianas em $\mathbb{R}^2$, com **covariâncias diferentes** — é essa
diferença que dá vantagem ao QDA sobre o LDA.

$$X\mid Y=0 \sim N\!\left(\begin{bmatrix}0\\0\end{bmatrix},
   \begin{bmatrix}1{,}00 & 0{,}75\\ 0{,}75 & 1{,}00\end{bmatrix}\right),
  \qquad
  X\mid Y=1 \sim N\!\left(\begin{bmatrix}2{,}20\\0{,}77\end{bmatrix},
   \begin{bmatrix}1{,}60 & -0{,}85\\ -0{,}85 & 0{,}70\end{bmatrix}\right).$$

Como conhecemos as densidades, o classificador de Bayes é imediato: com prioris
iguais, ele decide por $Y=1$ quando $f_1(x) > f_0(x)$.

In [ ]:
S0 = np.array([[1.0, 0.75], [0.75, 1.0]])
S1 = np.array([[1.6, -0.85], [-0.85, 0.7]])
mu0 = np.array([0.0, 0.0])
mu1 = np.array([2.2, 2.2 * 0.35])


def gera(n, rng):
    n0 = n // 2
    X = np.vstack([rng.multivariate_normal(mu0, S0, size=n0),
                   rng.multivariate_normal(mu1, S1, size=n - n0)])
    y = np.r_[np.zeros(n0), np.ones(n - n0)].astype(int)
    return X, y


rng = np.random.default_rng(2026)
X_grande, y_grande = gera(200000, rng)

f0 = multivariate_normal(mu0, S0).pdf(X_grande)
f1 = multivariate_normal(mu1, S1).pdf(X_grande)                 # (a) e (b)

decisao_bayes = (f1 > f0).astype(int)                             # (c)
erro_bayes = np.mean(decisao_bayes != y_grande)

print(f"erro de Bayes = {erro_bayes:.4f}   (acuracia maxima = {1 - erro_bayes:.4f})")

Deve imprimir `erro de Bayes = 0.0958   (acuracia maxima = 0.9042)`.

Guarde $0{,}9042$: é o teto. Qualquer método que chegue perto disso está
esgotando a informação que existe nos dados, e não há o que melhorar em modelagem
— só se pode reduzir esse número mudando as **covariáveis**, não o classificador.

Note também que o erro de Bayes é quase 10% mesmo com as duas nuvens visivelmente
separadas: as gaussianas se sobrepõem, e sobreposição é erro irredutível. É o
análogo em classificação do $\sigma^2$ da Aula 01.

> **Sua vez.** Desenhe as duas nuvens (500 pontos) num diagrama de dispersão, com
> cores diferentes por classe. Dá para ver por que a fronteira ótima não é uma
> reta?

---
## Exercício 2 — quatro classificadores, uma população

Agora esqueça que conhecemos as densidades e estime tudo a partir de 200
observações. Compare os quatro classificadores da aula contra o teto do
Exercício 1.

In [ ]:
rng = np.random.default_rng(2026)
X_tr, y_tr = gera(200, rng)
X_te, y_te = gera(20000, rng)

classificadores = [
    ("LDA",        LinearDiscriminantAnalysis()),                 # (a)
    ("QDA",        QuadraticDiscriminantAnalysis()),              # (b)
    ("GaussianNB", GaussianNB()),
    ("logistica",  LogisticRegression()),
]

for nome, modelo in classificadores:
    acuracia = modelo.fit(X_tr, y_tr).score(X_te, y_te)         # (c) e (d)
    print(f"{nome:12s} acuracia de teste {acuracia:.4f}")

# o teto, recalculado no mesmo conjunto de teste
otimo = (multivariate_normal(mu1, S1).pdf(X_te)
         > multivariate_normal(mu0, S0).pdf(X_te)).astype(int)
print(f"{'Bayes':12s} acuracia de teste {np.mean(otimo == y_te):.4f}")

Deve imprimir:

```
LDA          acuracia de teste 0.8627
QDA          acuracia de teste 0.9025
GaussianNB   acuracia de teste 0.8480
logistica    acuracia de teste 0.8716
Bayes        acuracia de teste 0.9023
```

O QDA chega a $0{,}9025$ contra os $0{,}9023$ do classificador de Bayes: ele
**alcança o teto** com 200 observações. Não é sorte nem virtude do algoritmo — é
que o QDA é o modelo *correto* para esta população. Ele supõe exatamente o que
usamos para gerar os dados (duas gaussianas com covariâncias distintas), e com
$n=200$ estima os 11 parâmetros com precisão suficiente.

Os outros três pagam pelo que supõem errado:

- o **LDA** ($0{,}8627$) impõe covariância comum, e as duas são bem diferentes:
  a fronteira reta não consegue acompanhar a curva verdadeira;
- o **GaussianNB** ($0{,}8480$) impõe covariâncias diagonais, e as correlações
  aqui são fortes ($+0{,}75$ e $-0{,}80$) e de sinais opostos — é o cenário do
  Exercício 4 da Lista Teórica 07, em versão suave;
- a **logística** ($0{,}8716$) não supõe nada sobre a distribuição de $X$, só que
  a fronteira é linear. Ela vence o LDA, que faz a mesma suposição de fronteira
  **mais** a suposição gaussiana — suposição que aqui não ajuda, porque a
  fronteira verdadeira não é reta.

Vale reter o padrão: **quando a suposição do modelo generativo é a verdade, o
plug-in encosta no teto; quando ela é falsa, o discriminativo costuma perder
menos.**

---
## Exercício 3 — LDA contra QDA, em função de $n$ e de $p$

A Lista Teórica 07 contou os parâmetros: em $p=2$ o QDA custa 3 a mais que o LDA;
em $p=10$, custa 55 a mais. Vamos ver o efeito disso.

A população em $\mathbb{R}^p$ é outra, construída do mesmo jeito para qualquer $p$:
duas gaussianas com covariâncias fixas diferentes, e médias separadas nas três
primeiras coordenadas. Em $p=2$, portanto, ela **não** é a população dos
Exercícios 1 e 2.

In [ ]:
_COVS = {}


def covariancia_fixa(d, qual):
    """Mesma matriz em toda chamada, para a populacao nao mudar entre repeticoes."""
    if (d, qual) not in _COVS:
        g = np.random.default_rng(1000 + qual)
        A = g.normal(size=(d, d))
        _COVS[(d, qual)] = (A @ A.T) / d + np.eye(d) * 0.5
    return _COVS[(d, qual)]


def gera_d(n, d, rng):
    n0 = n // 2
    mu = np.zeros(d)
    mu[:min(3, d)] = [1.6, 1.0, 0.7][:min(3, d)]
    X = np.vstack([rng.multivariate_normal(np.zeros(d), covariancia_fixa(d, 0), size=n0),
                   rng.multivariate_normal(mu, covariancia_fixa(d, 1), size=n - n0)])
    y = np.r_[np.zeros(n0), np.ones(n - n0)].astype(int)
    return X, y

In [ ]:
ns = np.array([20, 30, 50, 100, 300, 1000])

for d in (2, 10):
    X_teste, y_teste = gera_d(20000, d, np.random.default_rng(99))
    rng = np.random.default_rng(2026)
    media_lda, media_qda = [], []

    for n in ns:
        acc_lda, acc_qda = [], []
        for _ in range(60):
            X, y = gera_d(int(n), d, rng)
            # com n pequeno e d grande, a covariancia do QDA pode ser singular
            for Modelo, acc in ((LinearDiscriminantAnalysis, acc_lda),
                                (QuadraticDiscriminantAnalysis, acc_qda)):
                try:
                    acc.append(Modelo().fit(X, y).score(X_teste, y_teste))
                except Exception:
                    acc.append(np.nan)
        media_lda.append(np.nanmean(acc_lda))                     # (a)
        media_qda.append(np.nanmean(acc_qda))

    media_lda, media_qda = np.array(media_lda), np.array(media_qda)
    virada = ns[int(np.argmax(media_qda > media_lda))]            # (b) primeiro n em que o QDA passa

    print(f"d={d}:")
    print(f"   n   = {ns.tolist()}")
    print(f"   LDA = {media_lda.round(4).tolist()}")
    print(f"   QDA = {media_qda.round(4).tolist()}")
    print(f"   QDA passa a ganhar em n = {virada}")

Deve imprimir:

```
d=2:
   n   = [20, 30, 50, 100, 300, 1000]
   LDA = [0.8602, 0.8681, 0.8724, 0.8775, 0.8785, 0.8793]
   QDA = [0.8508, 0.8669, 0.8737, 0.8806, 0.8824, 0.8834]
   QDA passa a ganhar em n = 50
d=10:
   n   = [20, 30, 50, 100, 300, 1000]
   LDA = [0.7053, 0.7422, 0.7773, 0.8013, 0.8175, 0.8241]
   QDA = [nan, 0.703, 0.7896, 0.844, 0.8792, 0.8905]
   QDA passa a ganhar em n = 50
```

O ponto de virada é **o mesmo nas duas dimensões**, $n=50$ — e é por isso que ele
é a informação menos útil da tabela. O que muda é o **tamanho** do efeito:

| | $p=2$ | $p=10$ |
|---|---|---|
| vantagem do LDA em $n=30$ | $+0{,}0012$ | $+0{,}0392$ |
| vantagem do QDA em $n=1000$ | $+0{,}0041$ | $+0{,}0664$ |

Em $p=10$ a escolha importa muito, nos dois sentidos. O `nan` em $n=20$ é a conta
de parâmetros levada ao extremo: são 10 observações por classe em
$\mathbb{R}^{10}$, a covariância amostral de cada classe tem posto no máximo 9, e o
`scikit-learn` se recusa a ajustar o QDA — o `try` da célula registra a falha como
`nan`. Com $n=30$ ele já ajusta, mas fica 4 pontos atrás do LDA. Com $n=1000$ ele
recupera a estrutura verdadeira e ganha 6,6 pontos.

Em $p=2$ a diferença não passa de um ponto percentual em toda a tabela — mas não
por ser $p=2$. A matriz extra custa só 3 parâmetros, então o QDA tem pouco a
**perder**; e, nesta população, as duas classes têm quase a mesma correlação
($-0{,}47$ e $-0{,}45$), então ele também tem pouco a **ganhar**: mesmo com
$n=1000$, são 0,4 ponto. A população dos Exercícios 1 e 2 também tem $p=2$, e lá o
QDA ganha do LDA por 4 pontos com $n=200$, porque as correlações têm sinais opostos.

A leitura prática: **a regra "prefira LDA com poucos dados" só morde quando $p$ é
grande**, porque é aí que a matriz extra custa caro. Com $p$ pequeno o QDA sai
barato, e o que decide é se as covariâncias das classes de fato diferem.

---
## Exercício 4 — e num banco de verdade?

Nos exercícios anteriores nós geramos os dados, então sabíamos qual modelo estava
certo. No `breast_cancer` — 569 tumores, 30 medidas de imagem, resposta binária
benigno/maligno — ninguém sabe.

Um detalhe prático antes: as 30 medidas são quase colineares (raio, perímetro e área
do mesmo tumor), e o `QuadraticDiscriminantAnalysis` das versões recentes do
`scikit-learn` recusa o ajuste nesses dados. Use `reg_param=1e-4`, que puxa as
covariâncias levemente na direção da identidade.

In [ ]:
dados = load_breast_cancer()
X_bc, y_bc = dados.data, dados.target

print(f"n = {X_bc.shape[0]}, d = {X_bc.shape[1]}, "
      f"positivos = {100 * y_bc.mean():.1f}%")

cv = skm.StratifiedKFold(5, shuffle=True, random_state=2026)      # (a)

modelos = [
    ("LDA",        LinearDiscriminantAnalysis()),
    ("QDA",        QuadraticDiscriminantAnalysis(reg_param=1e-4)),        # (b)
    ("GaussianNB", GaussianNB()),
    ("logistica",  Pipeline([("escala", StandardScaler()),
                             ("modelo", LogisticRegression(max_iter=5000))])),
]

for nome, modelo in modelos:
    acuracia = skm.cross_val_score(modelo, X_bc, y_bc, cv=cv).mean()   # (c)
    print(f"{nome:12s} acuracia (CV) {acuracia:.4f}")

Deve imprimir `n = 569, d = 30, positivos = 62.7%` e:

```
LDA          acuracia (CV) 0.9596
QDA          acuracia (CV) 0.9508
GaussianNB   acuracia (CV) 0.9385
logistica    acuracia (CV) 0.9772
```

A ordem se inverte em relação ao Exercício 2. Aqui **a logística ganha**, e o QDA —
campeão absoluto na população sintética — fica em terceiro.

São dois motivos, e o segundo é o do Exercício 3. Primeiro, na população sintética o
QDA estava *certo* por construção; aqui não há nenhuma razão para as 30 medidas de
imagem serem gaussianas dentro de cada classe, e várias delas são claramente
assimétricas (`area`, `perimeter`). A logística não supõe nada sobre a distribuição
de $X$ — só sobre a fronteira — e por isso não sofre quando essa suposição falha.
Segundo, com $p=30$ o QDA precisa estimar $2\cdot 30\cdot 31/2 = 930$ parâmetros de
covariância a partir de cerca de 455 observações de treino por dobra — e só uns 170
são malignos. Ele ainda vai bem ($0{,}9508$), mas só chega a ser ajustado graças ao
`reg_param`; o LDA, com $465$, o supera.

O `GaussianNB` fica em último por outro motivo: raio, perímetro e área do mesmo
tumor são quase a mesma medida — correlação acima de $0{,}99$ dentro de cada
classe —, e ele os conta como três evidências independentes.

E note por que a padronização aparece só na logística: LDA, QDA e naive Bayes
estimam a covariância dos próprios dados, e a escala de cada coluna sai na conta;
a logística com penalização, não — é o Exercício 2(c) da Lista Teórica 02.

> **Sua vez.** A hipótese do LDA é que as duas classes têm a **mesma** matriz de
> covariância. Compare, coluna a coluna, a variância das medidas dentro dos tumores
> malignos com a variância dentro dos benignos — por exemplo, calculando a razão
> entre as duas para cada uma das 30 medidas. A hipótese parece razoável? E, se não
> parece, por que o LDA ganhou do QDA mesmo assim?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | o erro de Bayes desta população é 0,0958 — o teto de acurácia é 0,9042 |
| 2 | o QDA **alcança o teto** com $n=200$ (0,9025 contra 0,9023): ele é o modelo correto |
| 2 | a logística bate o LDA (0,8716 contra 0,8627), mesmo os dois traçando retas |
| 3 | em $p=10$ o QDA nem ajusta com $n=20$ e perde 4 pontos em $n=30$; em $p=2$ a escolha muda menos de 1 ponto |
| 4 | no `breast_cancer` a ordem se inverte: a logística ganha e o QDA cai para terceiro, e só ajusta com `reg_param` |

**A seguir.** A Aula 08 pega a segunda metade do problema. Todos os números desta
lista foram acurácias, e a população estava equilibrada. Quando 10% das
observações são positivas, a acurácia deixa de dizer qualquer coisa útil.